# 23 · Context Engineering：交给 LLM 之前

> 检索 ≠ 全塞。**召回的 10 个 chunk 直接全扔给 LLM 是最常见的偷懒**。Context Engineering 负责在“检索结果”与“最终 Prompt”之间加一层精加工。

**本文件覆盖知识点**：Context Selection / Compression / Filtering / Deduplication / Ordering / Truncation

```text
Retrieval → Rerank → 去重 → 过滤 → 压缩 → 排序 → LLM
```

In [ ]:
# ===== 本课共用：真调 LLM 做「说明 / 演示」的小助手 =====
# 凡某个知识点能靠“真调一次大模型”当场讲清 / 演示的，下面的 cell 都用 _llm_live()
# 真调 qwen-plus 并打印模型输出作为说明；只有在项目根 .env 配了 DASHSCOPE_API_KEY 时才真调，
# 没配置就打印一段固定的演示样例，保证整个 notebook 不联网也能完整读下来。
from dotenv import load_dotenv; load_dotenv()
import os
from dashscope import Generation

_KEY = os.getenv('DASHSCOPE_API_KEY', '').strip()
_HAS_KEY = bool(_KEY) and '你的' not in _KEY

def _llm_live(prompt, fallback, system='你是资深 RAG 讲师，回答精炼、结构清晰、尽量结合例子。', temperature=0.3, model='qwen-plus'):
    """真调一次 qwen-plus 并打印结果；无 Key 时打印 fallback 作为演示样例。返回模型文本或 None。"""
    if not _HAS_KEY:
        print('未在 .env 配置 DASHSCOPE_API_KEY，跳过实时调用。以下是固定演示样例（配置后自动变为实时输出）：')
        print(fallback)
        return None
    msgs = [{'role': 'system', 'content': system}, {'role': 'user', 'content': prompt}]
    try:
        r = Generation.call(model=model, messages=msgs, temperature=temperature, result_format='message', api_key=_KEY)
        if r.status_code == 200:
            text = r.output.choices[0].message.content
            print('—— 模型实时输出 ——')
            print(text)
            return text
        print('调用失败：', getattr(r, 'code', ''), getattr(r, 'message', ''))
    except Exception as e:
        print('调用异常：', e)
    print('fallback：')
    print(fallback)
    return None


## 1. 为什么不能“全部塞给 LLM”

1. **噪声干扰**：不相关的片段会“带偏”模型（尤其夹在开头附近）；
2. **成本**：token 即金钱，塞得越多越贵；
3. **效果**：研究 Lost-in-the-Middle 显示，被埋在中间的信息利用率差。

所以上线前要过这几道“闸”：

## 2. 五道闸

| 闸 | 做什么 | 手段 |
|----|--------|------|
| **去重 Dedup** | 同一事实被多篇文档复述，去重 | 文本/语义相似度聚类 |
| **过滤 Filter** | 丢掉阈值以下/类别不对的 | 相似度阈值、元数据、LLM 判官 |
| **压缩 Compress** | 长 chunk 掐头去尾留精华 | LLM 摘要、句子级抽取 |
| **排序 Order** | 把最相关的放最前/最后 | 按重排分降序 |
| **截断 Truncate** | 超出预算就砍 | 按 token 预算逐段放入 |

In [ ]:
# 知识点·真调说明：过滤 / 压缩 —— 五道闸里“LLM判官过滤 + LLM摘要压缩”长什么样
import json as _json
flt_q = '星云智能客服标准版怎么收费？'
flt_docs = [
    '标准版 998 元/月，含 5 个坐席与基础报表，超出后按条计费。',
    '免费试用版每月赠送 1000 条消息，无需绑定支付方式。',
    '企业版支持私有化部署与专属客服经理。',
    '报表支持导出 CSV / Excel 两种格式。',
]
print('① LLM 判官过滤 —— 召回 4 条，只保留“能直接回答收费问题”的')
flt_text = '\n'.join('[%s] %s' % (i + 1, t) for i, t in enumerate(flt_docs))
out1 = _llm_live(
    prompt='问题：%s\n\n候选片段：\n%s\n\n请逐条判断哪些片段与问题相关、值得保留给后续生成。' % (flt_q, flt_text),
    system='你是 RAG 的“上下文过滤判官”。规则：只保留能直接帮助回答该问题的片段；'
           '沾边但无助于本题、或讲其它版本/无关话题的一律剔除。只输出 JSON，禁止其它文字：'
           '{"keep_ids": [保留的编号], "drop_reasons": {"被剔除编号": "一句话原因"}}。',
    fallback='未配置 Key 的固定样例：\n'
             '{"keep_ids": [1], "drop_reasons": {"2": "讲免费试用版与收费无关", '
             '"3": "讲企业版不是标准版", "4": "讲报表导出格式与收费无关"}}',
    temperature=0.1,
)
if out1 is None:
    out1 = ('{"keep_ids": [1], "drop_reasons": {"2": "讲免费试用版与收费无关", '
            '"3": "讲企业版不是标准版", "4": "讲报表导出格式与收费无关"}}')
    print('（以上为固定样例；下面用样例演示 JSON 解析）')
try:
    o1 = _json.loads(out1[out1.find('{'): out1.rfind('}') + 1])
    print('保留:', o1['keep_ids'], ' 剔除:', o1.get('drop_reasons'))
except Exception as e:
    print('JSON 解析失败：', e, '—— 说明要收紧输出约束。')

print()
print('② LLM 摘要压缩 —— 长片段“掐头去尾”留能回答问题的要点')
long_doc = ('星云智能客服标准版按年订阅，年费 11,976 元（折合每月 998 元），包含 5 个坐席、自动应答、'
            '基础报表与工单系统。此外我们拥有完善的多轮对话引擎、智能知识库和全渠道接入能力，'
            '团队可提供 7×24 小时技术支持和专属客户成功经理，助您快速上线、持续优化运营效果。')
out2 = _llm_live(
    prompt='请把下面这段产品资料压缩成不超过 3 句的精炼版，只保留回答“怎么收费 / 含什么”所需的要点，'
           '删掉宣传话术与不必要修饰。\n资料：%s' % long_doc,
    system='你是 RAG 上下文压缩器。输出必须是资料里真实存在的信息，不要新增内容。',
    fallback='未配置 Key 的固定样例：\n'
             '标准版年费 11,976 元（每月 998 元），含 5 个坐席、自动应答、基础报表与工单系统。',
    temperature=0.2,
)
if out2 is None:
    out2 = '标准版年费 11,976 元（每月 998 元），含 5 个坐席、自动应答、基础报表与工单系统。'
    print('（以上为固定样例；以下用样例演示长度对比）')
print('原文 %d 字 → 压缩后 %d 字（送入 LLM 的 token 随之减少，成本下降）' % (len(long_doc), len(out2)))
print('→ 过滤丢“看似相关实则无关”的噪声、压缩省 token——这是生成前替 LLM 把关的两道闸。')

In [ ]:
# 一个朴素的 Context Builder：去重 + 截断到预算 token
def simple_context_builder(scored_docs, budget=600):
    """scored_docs: [(text, score), ...] 按分降序。返回最终拼接的上下文"""
    kept, seen, used = [], set(), 0
    for text, score in scored_docs:
        key = text[:50]                     # 粗糙去重：开头 50 字
        if key in seen:
            continue
        seen.add(key)
        est = len(text)                     # 以字符粗略当 token
        if used + est > budget:
            break                            # 预算截断
        kept.append((text, score)); used += est
    return kept

docs = [
    ('产品分三个版本，价格详见下文', 0.95),
    ('产品分三个版本，价格详见下文', 0.90),   # 重复 → 会被去重
    ('基础版 998 元/月', 0.88),
    ('支持私有化部署', 0.30),                  # 低相关噪音
]
kept = simple_context_builder(docs)
print('去重+预算截断后保留:')
for t, s in kept:
    print(f'  {s}  {t}')

In [ ]:
# 知识点·真调说明：去重 —— 开头50字粗去重拦不住“换了说法”的重复，语义判官才行
import json as _json
b_pairs = [
    ('产品分基础版、标准版、专业版三档，价格见后文。',
     '本产品设有基础、标准、专业三个版本，具体售价详见后文。'),
    ('产品分基础版、标准版、专业版三档，价格见后文。',
     '产品支持公有云与私有化两种部署方式。'),
]
print('上面朴素 builder 的开头 50 字去重：')
for i, (a, b) in enumerate(b_pairs, 1):
    print('  第%d对 开头50字是否相同=%s（相同才可能被它去重）' % (i, a[:50] == b[:50]))
print()
b_prompt = '\n'.join('第%d对：\nA. %s\nB. %s' % (i + 1, a, b) for i, (a, b) in enumerate(b_pairs))
out = _llm_live(
    prompt='请判断下面每一对片段是否在“陈述同一个事实（属于重复信息）”。\n%s' % b_prompt,
    system='你是 RAG 的语义去重判官：两段只要说的是同一件事就标 true（哪怕措辞完全不同），'
           '信息互补/各说各的就标 false。只输出 JSON：'
           '{"pairs": [{"duplicate": true或false, "reason": "一句话"}]}。禁止其它文字。',
    fallback='未配置 Key 的固定样例：\n'
             '{"pairs": [{"duplicate": true, "reason": "都说产品分三档且价格见后文，措辞不同实为同一事实"}, '
             '{"duplicate": false, "reason": "前句讲分档定价，后句讲部署方式，信息互补"}]}',
    temperature=0.1,
)
if out is None:
    out = ('{"pairs": [{"duplicate": true, "reason": "都说产品分三档且价格见后文，措辞不同实为同一事实"}, '
           '{"duplicate": false, "reason": "前句讲分档定价，后句讲部署方式，信息互补"}]}')
    print('（以上为固定样例；下面用样例演示 JSON 解析）')
try:
    objs = _json.loads(out[out.find('{'): out.rfind('}') + 1])['pairs']
    for i, o in enumerate(objs, 1):
        verdict = '重复信息，应去重' if o['duplicate'] else '信息互补，两个都保留'
        print('  第%d对 → %s（%s）' % (i, verdict, o.get('reason', '')))
    print('→ 换了说法的语义重复，开头截断式哈希抓不到；去重省下的正是送入 LLM 的 token 预算。')
except Exception as e:
    print('JSON 解析失败：', e, '—— 说明输出约束不够严。')

## 3. Context 的顺序怎么放

- 相关片段**放开头**最容易让模型注意到；
- 全放开头有时反而浪费——长文档场景试“最重要的在首尾，次要在中间”。

进阶上下文技术（**给片段补上下文、父子块、句窗**）见下一课（24）。

## 小结

- 别把召回结果“一把梭”喂给 LLM；
- 五道闸：**去重→过滤→压缩→排序→截断**；
- 上下文质量直接决定生成质量，也直接关系 token 成本。